# Overlap between the activity and the pose dataset

How much do the two training datasets share? Activity dataset: Kinodata-3D complexes with a pIC50 and a predicted RMSD ≤ 10 Å; pose dataset: cross-docked poses with a true RMSD ≤ 10 Å (the data the models are trained on). Overlap is counted at the level of ligands, scaffolds, kinases and exact kinase–ligand pairs:

- **Ligand:** the connectivity block of the InChIKey (RDKit), so that stereo and protonation variants of the same compound count as one ligand.
- **Scaffold:** two definitions. The *Bemis–Murcko scaffold* (ring systems and the linkers between them) and the *generic Bemis–Murcko scaffold*, in which every atom is replaced by carbon and every bond by a single bond. The generic scaffold is the one used to group the ligands in the scaffold split.
- **Kinase:** UniProt accession in the activity dataset and KLIFS kinase name in the pose dataset. A kinase counts as shared when one of its KLIFS pocket sequences is identical to a pocket sequence of the other dataset at all positions resolved in both (at least 70 of the 85 positions).
- **Kinase–ligand pair:** the same ligand bound to the same kinase (matching pocket sequence) in both datasets.

**Data** (`data/dataset_overlap/`): `activity.csv.gz` and `pose.csv.gz`, one row per complex or pose of the training data (SMILES, generic Bemis–Murcko scaffold as used by the split, pocket id, kinase), and `pockets.csv` (KLIFS pocket sequences). Tables are written to `output/dataset_overlap/`.

In [1]:
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kinodata" / "__init__.py").exists())
DATA = REPO / "analysis_publication" / "data" / "dataset_overlap"
OUTPUT = REPO / "analysis_publication" / "output" / "dataset_overlap"
OUTPUT.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")
MIN_POCKET_OVERLAP = 70

activity = pd.read_csv(DATA / "activity.csv.gz")
pose = pd.read_csv(DATA / "pose.csv.gz")
pockets = pd.read_csv(DATA / "pockets.csv").set_index("pocket")["sequence"]
print(f"activity: {len(activity):,} complexes; pose: {len(pose):,} poses; {len(pockets)} distinct pocket sequences")

activity: 119,421 complexes; pose: 67,976 poses; 870 distinct pocket sequences


## Ligand keys and scaffolds

Computed once per distinct SMILES. Check: the generic Bemis–Murcko scaffold recomputed here must be the scaffold used by the split.

In [2]:
def ligand_info(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, None
    bm = MurckoScaffold.GetScaffoldForMol(mol)
    generic = Chem.CanonSmiles(Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(bm)))
    return Chem.MolToInchiKey(mol)[:14], Chem.MolToSmiles(bm), generic


info = {s: ligand_info(s) for s in pd.unique(pd.concat([activity["smiles"], pose["smiles"]]))}
for df in (activity, pose):
    df["ligand"] = df["smiles"].map(lambda s: info[s][0])
    df["scaffold_bm"] = df["smiles"].map(lambda s: info[s][1])
    df["scaffold_generic"] = df["smiles"].map(lambda s: info[s][2])
    df["pocket_sequence"] = df["pocket"].map(pockets)
for name, df in [("activity", activity), ("pose", pose)]:
    same = (df["scaffold_generic"] == df["scaffold"]).mean()
    print(f"{name}: SMILES that RDKit cannot parse: {df['ligand'].isna().sum()}; generic scaffold = split scaffold for {same:.2%}")

activity: SMILES that RDKit cannot parse: 0; generic scaffold = split scaffold for 99.99%
pose: SMILES that RDKit cannot parse: 0; generic scaffold = split scaffold for 100.00%


## Kinase matching by pocket sequence

In [3]:
def same_pocket(a, b, min_overlap=MIN_POCKET_OVERLAP):
    """Same kinase: identical residues wherever both KLIFS pocket sequences have one."""
    if not (isinstance(a, str) and isinstance(b, str)) or len(a) != len(b):
        return False
    both = [(x, y) for x, y in zip(a, b) if x not in "_-" and y not in "_-"]
    return len(both) >= min_overlap and all(x == y for x, y in both)


act_pockets, pose_pockets = activity["pocket"].dropna().unique(), pose["pocket"].dropna().unique()
match = {(a, b) for a in act_pockets for b in pose_pockets if same_pocket(pockets[a], pockets[b])}
act_matched = {a for a, _ in match}
pose_matched = {b for _, b in match}
kin_act_shared = activity.loc[activity["pocket"].isin(act_matched), "kinase"].unique()
kin_pose_shared = pose.loc[pose["pocket"].isin(pose_matched), "kinase"].unique()
print(f"activity kinases (UniProt) with a pocket found in the pose dataset: {len(kin_act_shared)} of {activity['kinase'].nunique()}")
print(f"pose kinases (KLIFS name) with a pocket found in the activity dataset: {len(kin_pose_shared)} of {pose['kinase'].nunique()}")

activity kinases (UniProt) with a pocket found in the pose dataset: 113 of 248
pose kinases (KLIFS name) with a pocket found in the activity dataset: 113 of 124


## Kinase–ligand pairs

A pair is shared when the same ligand occurs with matching pockets in both datasets. Counted as distinct (ligand, activity pocket) combinations; the complexes and poses that belong to shared pairs are counted as well.

In [4]:
shared_ligands = set(activity["ligand"].dropna()) & set(pose["ligand"].dropna())
a = activity[activity["ligand"].isin(shared_ligands)].reset_index()
p = pose[pose["ligand"].isin(shared_ligands)].reset_index()
cand = a.merge(p, on="ligand", suffixes=("_act", "_pose"))
cand = cand[[(x, y) in match for x, y in zip(cand["pocket_act"], cand["pocket_pose"])]]
n_pairs = cand[["ligand", "pocket_act"]].drop_duplicates().shape[0]
n_act_in_pairs = cand["index_act"].nunique()
n_pose_in_pairs = cand["index_pose"].nunique()
print(f"shared kinase-ligand pairs: {n_pairs:,}; activity complexes in shared pairs: {n_act_in_pairs:,} "
      f"({n_act_in_pairs / len(activity):.2%}); poses in shared pairs: {n_pose_in_pairs:,} ({n_pose_in_pairs / len(pose):.0%})")

shared kinase-ligand pairs: 562; activity complexes in shared pairs: 611 (0.51%); poses in shared pairs: 22,069 (32%)


## Overlap table

Distinct items per dataset, the number shared, and the shared fraction of each dataset.

In [5]:
def row(level, act_items, pose_items, shared=None, note=""):
    act_items, pose_items = set(pd.Series(list(act_items)).dropna()), set(pd.Series(list(pose_items)).dropna())
    n_shared = len(act_items & pose_items) if shared is None else shared
    return {"level": level, "activity dataset": len(act_items), "pose dataset": len(pose_items), "shared": n_shared,
            "shared / activity (%)": round(100 * n_shared / len(act_items), 1), "shared / pose (%)": round(100 * n_shared / len(pose_items), 1),
            "note": note}


table = pd.DataFrame([
    {"level": "complexes / poses (RMSD ≤ 10 Å)", "activity dataset": len(activity), "pose dataset": len(pose),
     "shared": f"{n_act_in_pairs:,} complexes / {n_pose_in_pairs:,} poses in shared pairs",
     "shared / activity (%)": round(100 * n_act_in_pairs / len(activity), 1), "shared / pose (%)": round(100 * n_pose_in_pairs / len(pose), 1), "note": ""},
    row("ligands (InChIKey connectivity)", activity["ligand"], pose["ligand"]),
    row("Bemis–Murcko scaffolds", activity["scaffold_bm"], pose["scaffold_bm"]),
    row("generic Bemis–Murcko scaffolds (scaffold split)", activity["scaffold_generic"], pose["scaffold_generic"]),
    {**row("kinases", activity["kinase"], pose["kinase"], shared=len(kin_act_shared), note="activity: UniProt; pose: KLIFS name; shared = matching pocket"),
     "shared / pose (%)": round(100 * len(kin_pose_shared) / pose["kinase"].nunique(), 1)},
    {"level": "kinase–ligand pairs", "activity dataset": "", "pose dataset": "", "shared": n_pairs,
     "shared / activity (%)": "", "shared / pose (%)": "", "note": "same ligand, matching pocket"},
]).set_index("level")
table.to_csv(OUTPUT / "overlap_table.csv")
table

,activity dataset,pose dataset,shared,shared / activity (%),shared / pose (%),note
level,,,,,,
complexes / poses (RMSD ≤ 10 Å),119421,67976,"611 complexes / 22,069 poses in shared pairs",0.5,32.5,
ligands (InChIKey connectivity),83635,1469,748,0.9,50.9,
Bemis–Murcko scaffolds,35701,1454,870,2.4,59.8,
generic Bemis–Murcko scaffolds (scaffold split),16349,980,768,4.7,78.4,
kinases,248,124,113,45.6,91.1,activity: UniProt; pose: KLIFS name; shared = ...
kinase–ligand pairs,,,562,,,"same ligand, matching pocket"


### Complexes covered by shared scaffolds

Share of the activity complexes and of the poses whose (generic or Bemis–Murcko) scaffold also occurs in the other dataset. Distinct scaffolds (table above) and complexes can differ a lot, because a few scaffolds hold many poses.

In [6]:
rows = []
for col, name in [("scaffold_generic", "generic Bemis–Murcko scaffold"), ("scaffold_bm", "Bemis–Murcko scaffold")]:
    sa, sp = set(activity[col].dropna()), set(pose[col].dropna())
    rows.append({"scaffold": name, "activity complexes with a scaffold found in the pose dataset (%)": round(100 * activity[col].isin(sp).mean(), 1),
                 "poses with a scaffold found in the activity dataset (%)": round(100 * pose[col].isin(sa).mean(), 1)})
coverage = pd.DataFrame(rows).set_index("scaffold")
coverage.to_csv(OUTPUT / "scaffold_coverage.csv")
coverage

,activity complexes with a scaffold found in the pose dataset (%),poses with a scaffold found in the activity dataset (%)
scaffold,,
generic Bemis–Murcko scaffold,25.9,88.6
Bemis–Murcko scaffold,9.7,75.6
